# Smart Query Assistant — Production AI Pipeline

A multi-source retrieval + decision-based answering pipeline with memory,  
built using **LangChain**, **LangGraph**, **Groq (LLaMA-3.3-70b)**, and **MCP (Wikipedia)**.

---

## Architecture

```
User Input (user_id, thread_id, query)
    │
    ▼
[thread_check]          ← detect new vs existing thread
    │
    ▼
[memory]                ← retrieve relevant past exchanges (FAISS)
    │
    ├──────────────────────────────────┐
    ▼                                  ▼
[youtube_retriever]            [wiki_retriever]     ◄── PARALLEL
(FAISS vector search)          (Wikipedia via MCP)
    │                                  │
    └──────────────┬───────────────────┘
                   ▼
               [merge]            ← deduplicate + combine
                   │
                   ▼
               [decision]         ← LLM returns strict JSON
                   │
       ┌───────────┼───────────┬───────────┐
       ▼           ▼           ▼           ▼
 [yt_answer] [wiki_answer] [hybrid]  [fallback]   ◄── CONDITIONAL
       └───────────┴───────────┴───────────┘
                   │
                   ▼
                [save]            ← persist Q&A to memory
                   │
                  END
```

---

## Stack

| Component | Technology |
|-----------|------------|
| LLM | Groq — `llama-3.3-70b-versatile` |
| Orchestration | LangGraph `StateGraph` |
| Tool Protocol | MCP (`mcp` + `langchain-mcp-adapters`) |
| Vector Store | FAISS + `all-MiniLM-L6-v2` embeddings |
| Knowledge Sources | YouTube transcripts (FAISS) · Wikipedia (MCP) |


---
## Step 1 — Environment Setup

This project runs inside the **`bluParrot`** conda environment.

### 1a. Activate the environment (run in your terminal, NOT in the notebook)

```bash
conda activate bluParrot
```

### 1b. Install the Jupyter kernel for bluParrot (one-time setup)

```bash
conda activate bluParrot
python -m ipykernel install --user --name bluParrot --display-name "Python (bluParrot)"
```

Then in Jupyter: **Kernel → Change Kernel → Python (bluParrot)**

### 1c. Install all dependencies

Run the cell below — all packages are already in `requirements.txt`.

In [ ]:
# Install all required packages into the active environment.
# Skip this cell if packages are already installed.
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"],
    check=True
)
print("✓ All packages installed.")

---
## Step 2 — Verify Environment

In [ ]:
# Verify that all critical packages are importable and show their versions.
import importlib

packages = [
    "langchain",
    "langchain_groq",
    "langchain_community",
    "langchain_huggingface",
    "langgraph",
    "mcp",
    "langchain_mcp_adapters",
    "faiss",
    "sentence_transformers",
    "wikipedia",
    "dotenv",
]

for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "n/a")
        print(f"  ✓  {pkg:<30} {ver}")
    except ImportError:
        print(f"  ✗  {pkg:<30} NOT FOUND — run Step 1")

---
## Step 3 — Configure API Key

The Groq API key is loaded from the `.env` file in the same directory.
It was pre-populated when the project was created.  
Run the cell below to confirm it is loaded correctly.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=".env")

key = os.getenv("GROQ_API_KEY", "")
if key:
    masked = key[:8] + "*" * (len(key) - 12) + key[-4:]
    print(f"✓ GROQ_API_KEY loaded: {masked}")
else:
    print("✗ GROQ_API_KEY not found. Add it to your .env file:")
    print('  GROQ_API_KEY=your_key_here')

---
## Step 4 — Understand the Pipeline Code

The entire pipeline lives in **`pipeline.py`** — a single, self-contained file.  
The same file acts as **two things**:

| Mode | How it's triggered | What it does |
|------|-------------------|--------------|
| **Pipeline** | `python pipeline.py` | Runs the LangGraph |
| **MCP Server** | `python pipeline.py --mcp-server` | Serves Wikipedia tools over stdio |

When the pipeline runs, it automatically spawns itself as the MCP server in a subprocess via `MultiServerMCPClient`.

### Key design decisions:

- **Wikipedia via MCP only** — no direct API calls; respects the MCP requirement.
- **Parallel retrieval** — LangGraph fans out from `memory` to `youtube_retriever` and `wiki_retriever` concurrently, merges state at `merge`.
- **Decision node** — LLM returns `{"decision": "youtube|wiki|hybrid|fallback"}` in strict JSON; parsed safely with a fallback.
- **Memory grows per turn** — each Q&A is embedded and added to FAISS for future semantic retrieval.

View the source:

In [ ]:
# Print the first 80 lines of the pipeline to show the architecture diagram.
with open("pipeline.py", "r") as f:
    lines = f.readlines()

print("".join(lines[:80]))

---
## Step 5 — Import the Pipeline

In [ ]:
# The pipeline module is imported here.
# Because pipeline.py checks sys.argv for '--mcp-server' at the top,
# importing it is safe — it will NOT start an MCP server.

import asyncio
from pipeline import run_pipeline

print("✓ Pipeline imported successfully.")

---
## Step 6 — Run the Test Case (from Requirements)

In [ ]:
# Required test case:
#   user_id   = "user_1"
#   thread_id = "thread_1"
#   query     = "What is overfitting?"

result = await run_pipeline(
    user_id   = "user_1",
    thread_id = "thread_1",
    query     = "What is overfitting?",
)

print("\n=== Final Result ===")
print(f"Source : {result['selected_source'].upper()}")
print(f"Answer : {result['answer']}")

---
## Step 7 — Run Custom Queries

In [ ]:
# Try a different query — watch how the Decision Node picks the source.

result2 = await run_pipeline(
    user_id   = "user_1",
    thread_id = "thread_1",   # same thread → memory context will be retrieved
    query     = "How does dropout prevent overfitting?",
)

print(f"Source : {result2['selected_source'].upper()}")
print(f"Answer : {result2['answer']}")

In [ ]:
# Try a topic that is likely to route to Wikipedia (general knowledge).

result3 = await run_pipeline(
    user_id   = "user_1",
    thread_id = "thread_2",   # new thread → no memory
    query     = "What is the history of artificial intelligence?",
)

print(f"Source : {result3['selected_source'].upper()}")
print(f"Answer : {result3['answer']}")

In [ ]:
# Try a niche query that neither source might cover → should route to FALLBACK.

result4 = await run_pipeline(
    user_id   = "user_1",
    thread_id = "thread_3",
    query     = "What was the weather in Tokyo on January 5, 1998?",
)

print(f"Source : {result4['selected_source'].upper()}")
print(f"Answer : {result4['answer']}")

---
## Step 8 — Run from Terminal (Alternative)

You can also run the pipeline directly from the terminal without this notebook:

```bash
conda activate bluParrot
cd "/Users/tejas/Documents/BluParrot/Smart Query Assistant"
python pipeline.py
```

The test case (`user_1 / thread_1 / "What is overfitting?"`) runs automatically  
from the `if __name__ == "__main__":` block at the bottom of `pipeline.py`.

---
## Step 9 — Visualise the LangGraph

LangGraph can render the compiled graph as a Mermaid diagram.

In [ ]:
# Build a minimal graph just to render it (no LLM/MCP calls needed).

from pipeline import (
    build_embeddings, build_youtube_store, build_memory_store, build_llm,
    build_graph, MultiServerMCPClient
)
import sys

# We need a placeholder wiki_tool for graph construction.
# Build a real one via MCP so the graph is identical to what runs in production.
import pipeline as _pl

_embeddings    = build_embeddings()
_youtube_store = build_youtube_store(_embeddings)
_memory_store  = build_memory_store(_embeddings)
_llm           = build_llm()

_mcp_client = MultiServerMCPClient({
    "wikipedia": {
        "command":   sys.executable,
        "args":      [_pl.__file__, "--mcp-server"],
        "transport": "stdio",
    }
})
_tools    = await _mcp_client.get_tools()
_wiki_tool = next(t for t in _tools if t.name == "search_wikipedia")

_graph = build_graph(_llm, _youtube_store, _memory_store, _wiki_tool)

# Print the Mermaid diagram source
print(_graph.get_graph().draw_mermaid())

Paste the output above into [mermaid.live](https://mermaid.live) to see the full graph rendered visually.

---

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `ModuleNotFoundError` | Run Step 1 (pip install) |
| `GROQ_API_KEY not set` | Check `.env` file in the project directory |
| `search_wikipedia not found` | Ensure `mcp` and `langchain-mcp-adapters` are installed |
| Kernel not found | Run `python -m ipykernel install --user --name bluParrot` in terminal |
| `await` error outside async | Use Jupyter (has implicit event loop) or wrap in `asyncio.run(...)` |

> **Note on `await` in notebooks:**  
> Jupyter notebooks support top-level `await` natively.  
> If running in a plain Python script, replace `await run_pipeline(...)` with `asyncio.run(run_pipeline(...))`.
